In [18]:
import torch
from torchvision import transforms as T
from torch.utils.data import DataLoader, Dataset, Subset
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchmetrics

import pytorch_lightning as L
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from PIL import Image

In [ ]:
class ChessDataset(Dataset):
    def __init__(self, csv_file, img_dir='images', transform=None, is_test=False):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

        if not self.is_test and 'label' in self.df.columns:
            self.classes = sorted(self.df['label'].unique())
            self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
            self.idx_to_class = {i: cls_name for i, cls_name in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        img_path = (
            row['image_path']
            if self.img_dir is None
            else os.path.join(self.img_dir, row['image_path'])
        )
        image = Image.open(img_path).convert('RGB')
        img_id = row['id']

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, img_id
        else:
            label = self.class_to_idx[row['label']]
            return image, torch.tensor(label, dtype=torch.long)


class ChessDataModule(L.LightningDataModule):
    def __init__(
            self, train_csv, test_csv, batch_size=32, img_size=224, val_split=0.2
    ):
        super().__init__()
        self.train_csv = train_csv
        self.test_csv = test_csv
        self.batch_size = batch_size
        self.img_size = img_size
        self.val_split = val_split

        self.transform_train = T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(),
            T.RandomRotation(15),
            T.ColorJitter(
                brightness=0.2, contrast=0.2, saturation=0.2
            ),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
            )
        ])

        self.transform_val = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
            )
        ])

    def setup(self, stage=None):
        full_dataset = ChessDataset(self.train_csv, transform=self.transform_train)
        self.num_classes = len(full_dataset.classes)
        self.class_to_idx = full_dataset.class_to_idx
        self.idx_to_class = full_dataset.idx_to_class

        train_idx, val_idx = train_test_split(
            range(len(full_dataset)),
            test_size=self.val_split,
            random_state=42,
            stratify=full_dataset.df['label']
        )

        self.train_dataset = Subset(full_dataset, train_idx)

        val_dataset_eval = ChessDataset(
            self.train_csv, transform=self.transform_val
        )
        self.val_dataset = Subset(val_dataset_eval, val_idx)
        self.test_dataset = ChessDataset(
            self.test_csv, transform=self.transform_val, is_test=True
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=4,
            persistent_workers=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4,
            persistent_workers=True
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4,
            persistent_workers=True
        )

In [20]:
class ChessClassifier(L.LightningModule):
    def __init__(self, num_classes, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        self.backbone = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)

        self.criterion = nn.CrossEntropyLoss()

        self.train_acc = torchmetrics.Accuracy(
            task='multiclass', num_classes=num_classes
        )
        self.val_acc = torchmetrics.Accuracy(
            task='multiclass', num_classes=num_classes
        )

    def forward(self, x):
        return self.backbone(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        self.train_acc(preds,y)

        self.log(
            'train_loss',
            loss,
            on_step=True,
            on_epoch=True,
            prog_bar=True,
            logger=True
        )
        self.log(
            'train_acc',
            self.train_acc,
            on_step=False,
            on_epoch=True,
            prog_bar=True,
            logger=True
        )
        return loss

    def validation_step(self, batch, batch_idx):
        x ,y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        self.val_acc(preds, y)

        self.log(
            'val_loss', 
            loss, on_step=False, 
            on_epoch=True, 
            prog_bar=True, 
            logger=True
        )
        self.log(
            'val_acc',
            self.val_acc,
            on_step=False,
            on_epoch=True,
            prog_bar=True,
            logger=True
        )

        return loss

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.1, patience=2
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss'
            }
        }

In [21]:
dm = ChessDataModule(
    train_csv='train.csv', test_csv='test.csv', batch_size=32, img_size=224
)
dm.setup()

model = ChessClassifier(num_classes=dm.num_classes, lr=1e-3)

checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    dirpath='checkpoints/',
    filename='chess-resnet-{epoch:02d}-{val_loss:.4f}',
    save_top_k=1,
    mode='min'
)

early_stop_callback = EarlyStopping(
    monitor='val_loss', patience=5, mode='min', verbose=True
)

trainer = L.Trainer(
    max_epochs=10,
    accelerator='auto',
    devices=1,
    callbacks=[checkpoint_callback, early_stop_callback],
    precision='16-mixed'
)

trainer.fit(model, dm)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/moldo/Projects/MLCompete/Medium/PieseSah/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name      | Type               | Params | Mode  | FLOPs
----------------------------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/pytorch_lightning/loops/fit_loop.py:321: The number of training batches (13) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 0: 100%|██████████| 13/13 [00:01<00:00,  9.35it/s, v_num=2, train_loss_step=0.810, val_loss=14.00, val_acc=0.327, train_loss_epoch=1.110, train_acc=0.601]

Metric val_loss improved. New best score: 13.977


... (5 outputs left out)

Metric val_loss improved by 0.197 >= min_delta = 0.0. New best score: 0.298


Epoch 9: 100%|██████████| 13/13 [00:01<00:00,  9.69it/s, v_num=2, train_loss_step=0.0408, val_loss=0.536, val_acc=0.827, train_loss_epoch=0.112, train_acc=0.966]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 13/13 [00:01<00:00,  9.52it/s, v_num=2, train_loss_step=0.0408, val_loss=0.536, val_acc=0.827, train_loss_epoch=0.112, train_acc=0.966]


In [23]:
best_model_path = checkpoint_callback.best_model_path
model = ChessClassifier.load_from_checkpoint(best_model_path)

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

test_loader = dm.test_dataloader()
results = []

with torch.no_grad():
    for images, img_ids in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        for img_id, pred_idx in zip(img_ids, preds):
            label_name = dm.idx_to_class[pred_idx]
            results.append({
                'id': img_id,
                'label': label_name
            })

submission_df = pd.DataFrame(results)
submission_df.to_csv('submission.csv', index=False)